In [1]:
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel

# ===================== 1. 加载预训练BERT与Tokenizer =====================
# 中文基础BERT，无需微调、无需标注数据
model_name = "bert-base-chinese"
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertModel.from_pretrained(model_name)
bert_model.eval()  # 推理模式，关闭dropout

# 分词阈值：可根据文本调整，值越大切分越细，值越小词组越长
SPLIT_THRESHOLD = 0.7

# ===================== 2. 单句无监督分词核心函数 =====================
def bert_unsupervised_segment(text: str) -> list:
    """
    BERT 无监督中文分词（零标注）
    :param text: 原始中文句子
    :return: 分词列表
    """
    # 1. 字符级编码（保留每个汉字）
    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,  # 保留  、
        padding=False,
        truncation=False
    )
    input_ids = inputs["input_ids"]
    seq_len = input_ids.shape[1]

    # 2. 提取BERT最后一层字符向量（去掉首尾特殊符）
    with torch.no_grad():
        outputs = bert_model(**inputs)
        # last_hidden_state: [1, seq_len, hidden_dim]
        char_embeds = outputs.last_hidden_state[0][1:-1]  # 去掉 、

    # 3. 逐对计算相邻字符余弦相似度
    words = []
    current_word = [text[0]]

    for i in range(len(char_embeds) - 1):
        emb1 = char_embeds[i]
        emb2 = char_embeds[i + 1]
        # 计算余弦相似度
        sim = F.cosine_similarity(emb1, emb2, dim=0).item()

        if sim < SPLIT_THRESHOLD:
            # 相似度低：切分，保存当前词
            words.append("".join(current_word))
            current_word = [text[i + 1]]
        else:
            # 相似度高：合并
            current_word.append(text[i + 1])

    # 加入最后一个词
    if current_word:
        words.append("".join(current_word))
    return words

# ===================== 3. 测试演示 =====================
if __name__ == "__main__":
    test_sentences = [
        "人工智能改变生活",
        "今天天气非常晴朗",
        "我喜欢深度学习和自然语言处理",
        "北京大学位于北京市海淀区"
    ]

    for sent in test_sentences:
        seg_result = bert_unsupervised_segment(sent)
        print(f"原句：{sent}")
        print(f"分词：{' / '.join(seg_result)}\n")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

C:\Users\kim\.virtualenvs\cv2-AFZlAyVW\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kim\.cache\huggingface\hub\models--bert-base-chinese. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

原句：人工智能改变生活
分词：人 / 工 / 智能 / 改变 / 生活

原句：今天天气非常晴朗
分词：今 / 天天气非常晴 / 朗

原句：我喜欢深度学习和自然语言处理
分词：我 / 喜欢 / 深度学习 / 和 / 自 / 然 / 语言 / 处 / 理

原句：北京大学位于北京市海淀区
分词：北 / 京 / 大学 / 位于 / 北京市 / 海 / 淀 / 区

